### MiddleWare's
 * first we will Learn about summarization MiddleWare
### SummarizationMiddleware
* Summarizes conversation history when **token limits** are approached.

This middleware monitors message token counts and automatically summarizes older messages when a threshold is reached, preserving recent messages and maintaining context continuity by ensuring AI/Tool message pairs remain together

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import SystemMessage,HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
llmModel = ChatGroq(model="qwen/qwen3-32b")


agents = create_agent(
       model=llmModel,
       checkpointer=InMemorySaver(),
       middleware=[
           SummarizationMiddleware(
                model=llmModel,
                trigger = ("messages",10),
                keep = ("messages",5)

           )
       ]
)


In [2]:
config = {"configurable":{"thread_id":"test-1"}}

In [3]:
question = [
    "What is the capital of France?",
    "what is the capital of Germany?",
    "what is the capital of Italy?",
    "what is the capital of Spain?",
    "what is the capital of Portugal?",
    "what is the capital of Netherlands?",
]

for q in question:
    response = agents.invoke({"messages": [HumanMessage(content=q)]}, config=config)
    print(response)

{'messages': [HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}, id='23a215aa-166d-44d1-a7e3-9c6dde3aec7b'), AIMessage(content="<think>\nOkay, so I need to figure out what the capital of France is. Hmm, I remember from school that France is a country in Europe. Let me think. I know Paris is a big city there. Wait, isn't Paris the capital? I think I've heard that before. But maybe I should double-check. Sometimes countries have different cities as capitals. For example, I know that the capital of Germany is Berlin, and the capital of Italy is Rome. So, following that pattern, maybe France's capital is Paris. But wait, could there be any exceptions or something? Maybe there's a smaller city that's the capital. No, I think Paris is definitely the capital. I've seen movies and shows set in Paris that mention it being a major city in France. Also, the Eiffel Tower is in Paris, which is a famous landmark. Yeah, I'm pretty sure Paris is the capi

### MIDDELWARE'S 
**Human-in-the-loop**


In [4]:

def  read_mail(mail_id:str):
    """A tool to read mail id by its ID."""
    return f"Reading mail with ID: {mail_id}"

def send_mail(to:str, subject:str, body:str):
    """A tool to send mail to a recipient."""
    return f"Sending mail to {to} with subject: {subject} and body: {body}"


In [5]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
hl_agent = create_agent(
    model=llmModel,
    checkpointer=InMemorySaver(),
    tools=[read_mail, send_mail],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on = {
                "send_mail": {
                    "allowed_decisions":['approve','edit','reject']
                },
                "read_mail": False
            }
        )
    ]
)

In [6]:
config1 = {"configurable":{"thread_id":"test-5"}}

In [10]:
res = hl_agent.invoke({"messages": [HumanMessage(content="Please send an email to John with subject 'Meeting' and body 'Let's meet tomorrow.'")]}, config=config1, version="v2")

if res.interrupts :
    print("⏸️ Agent paused! Review the action:")
    for interrupt in res.interrupts:
        print(interrupt)
    decision = "approve"
    from langgraph.types import Command
    final_res = hl_agent.invoke(
        Command( resume ={
            "decisions":[{"type": decision}]
        }
        ),
        config=config1,
        version="v2"
    )
    print(final_res)
else:
    print(res)


⏸️ Agent paused! Review the action:
Interrupt(value={'action_requests': [{'name': 'send_mail', 'args': {'body': "Let's meet tomorrow.", 'subject': 'Meeting', 'to': 'John'}, 'description': 'Tool execution requires approval\n\nTool: send_mail\nArgs: {\'body\': "Let\'s meet tomorrow.", \'subject\': \'Meeting\', \'to\': \'John\'}'}], 'review_configs': [{'action_name': 'send_mail', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='435d2b527508bfd803657aea841082b6')
GraphOutput(value={'messages': [HumanMessage(content="Please send an email to John with subject 'Meeting' and body 'Let's meet tomorrow.'", additional_kwargs={}, response_metadata={}, id='41a22801-574a-405b-88c5-fa8c2ad4dd19'), AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to John. Let me check the available tools. There's a send_mail function. The parameters required are to, subject, and body. The user provided all three: to is John, subject is 'Meeting', and body i